# Stage 10B: tail-safe alignment gate and spatial audit

This standalone Colab notebook keeps the Stage 10 `loose × 0.2` signal, tests conservative caps and visible-prefix confidence gates, and validates them with both standard five-fold and spatial six-fold partitions. It never submits to Kaggle. CPU is sufficient.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess

REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
data_dir = drive_root / 'data'
artifact_dir = drive_root / 'artifacts'
assert (data_dir / 'train').is_dir(), f'Missing: {data_dir / "train"}'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
artifact_dir.mkdir(parents=True, exist_ok=True)
subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], check=True)


In [ ]:
ALIGNMENT_RUN_ID = 'stage10_multiscale_alignment_full_v001'
alignment_run = artifact_dir / ALIGNMENT_RUN_ID
if not (alignment_run / 'gate_summary.json').is_file():
    cutback_candidates = [artifact_dir / 'stage8_safe_cutback_gate_full_v002', artifact_dir / 'stage8_safe_cutback_gate_full_v001']
    cutback_run = next((p for p in cutback_candidates if (p / 'candidate_matrix.parquet').is_file()), None)
    assert cutback_run is not None, 'Stage 8 matrix is missing. Run notebook 190 once; 00_colab_setup is not required.'
    subprocess.run([
        'uv', 'run', 'rogii-alignment',
        '--config', 'configs/experiment/stage10_multiscale_alignment.yaml',
        '--cutback-run', str(cutback_run), '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir), '--run-id', ALIGNMENT_RUN_ID,
    ], cwd=repo_dir, check=True)
print('Using:', alignment_run)


In [ ]:
RUN_ID = 'stage10b_tail_safe_alignment_full_v001'
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'gate_summary.json').is_file():
    command = [
        'uv', 'run', 'rogii-alignment-gate',
        '--config', 'configs/experiment/stage10b_tail_safe_alignment.yaml',
        '--alignment-run', str(alignment_run), '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir), '--run-id', RUN_ID,
    ]
    process = subprocess.Popen(command, cwd=repo_dir, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
    if process.wait() != 0:
        raise RuntimeError(f'Stage 10B failed with exit code {process.returncode}')
else:
    print('Reusing completed run:', run_dir)


In [ ]:
s = json.loads((run_dir / 'gate_summary.json').read_text())
{
    'promoted': s['promoted'],
    'base_rmse': s['base_metrics']['pooled_rmse'],
    'nested_candidate_rmse': s['candidate_metrics']['pooled_rmse'],
    'rmse_delta': s['pooled_rmse_delta'],
    'bootstrap_95pct': [s['bootstrap']['ci_2_5'], s['bootstrap']['ci_97_5']],
    'improved_folds': f"{s['improved_folds']}/{len(s['fold_deltas'])}",
    'fold_deltas': s['fold_deltas'],
    'spatial_delta': s['spatial']['pooled_rmse_delta'],
    'spatial_fold_deltas': s['spatial']['fold_deltas'],
    'gates': s['gates'],
    'inference_profile': s['inference_profile'],
    'standard_selections': s['standard_selections'],
    'spatial_selections': s['spatial_selections'],
    'profile_report': s['profile_report'],
}
